In [16]:
%matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt
import catboost
import lightgbm
import pandas as pd
import xgboost
import optuna
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
import xgboost as xgb
import catboost as cat
import lightgbm as lgb
import time
import smogn
import tensorflow as tf
import holidays
from datetime import datetime
pd.set_option('display.max_columns', None)


In [17]:
# load test.csv and train.csv as pandas dataframes
LOCAL = 1 
if LOCAL:
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
else:
    train = pd.read_csv('/kaggle/input/c2cccc/train.csv')
    test = pd.read_csv('/kaggle/input/c2cccc/test.csv')

In [18]:
DATAIMPUTEAVG = False
DATABALANCE = True
DATAMULT = False 
if DATAMULT:
    train = pd.concat([train] * 5, ignore_index=True)

In [19]:
train.head(), train.shape, test.head(), test.shape

(   id        date country              store             product  num_sold
 0   0  2010-01-01  Canada  Discount Stickers   Holographic Goose       NaN
 1   1  2010-01-01  Canada  Discount Stickers              Kaggle     973.0
 2   2  2010-01-01  Canada  Discount Stickers        Kaggle Tiers     906.0
 3   3  2010-01-01  Canada  Discount Stickers            Kerneler     423.0
 4   4  2010-01-01  Canada  Discount Stickers  Kerneler Dark Mode     491.0,
 (230130, 6),
        id        date country              store             product
 0  230130  2017-01-01  Canada  Discount Stickers   Holographic Goose
 1  230131  2017-01-01  Canada  Discount Stickers              Kaggle
 2  230132  2017-01-01  Canada  Discount Stickers        Kaggle Tiers
 3  230133  2017-01-01  Canada  Discount Stickers            Kerneler
 4  230134  2017-01-01  Canada  Discount Stickers  Kerneler Dark Mode,
 (98550, 5))

In [20]:
# find all unique values in all columns and also the count of unique values
for col in train.columns:
    print(col, train[col].nunique())
    print(train[col].unique())
    print("dtype: ", train[col].dtype)
    print("")

id 230130
[     0      1      2 ... 230127 230128 230129]
dtype:  int64

date 2557
['2010-01-01' '2010-01-02' '2010-01-03' ... '2016-12-29' '2016-12-30'
 '2016-12-31']
dtype:  object

country 6
['Canada' 'Finland' 'Italy' 'Kenya' 'Norway' 'Singapore']
dtype:  object

store 3
['Discount Stickers' 'Stickers for Less' 'Premium Sticker Mart']
dtype:  object

product 5
['Holographic Goose' 'Kaggle' 'Kaggle Tiers' 'Kerneler'
 'Kerneler Dark Mode']
dtype:  object

num_sold 4037
[  nan  973.  906. ... 3446. 2266. 3996.]
dtype:  float64



In [21]:
print(train[train['num_sold'].isnull() | (train['num_sold'] == 0)].shape)
print(train[train.isnull().any(axis=1) | (train['id'].isnull()) | (train['date'].str.len() == 0)].shape)
print(test[test.isnull().any(axis=1) | (test['id'].isnull()) | (test['date'].str.len() == 0)].shape)
train[train['num_sold'].isnull() | (train['num_sold'] == 0)].head()

(8871, 6)
(8871, 6)
(0, 5)


,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
45,45,2010-01-01,Kenya,Discount Stickers,Holographic Goose,NaN
90,90,2010-01-02,Canada,Discount Stickers,Holographic Goose,NaN
135,135,2010-01-02,Kenya,Discount Stickers,Holographic Goose,NaN
180,180,2010-01-03,Canada,Discount Stickers,Holographic Goose,NaN


In [22]:
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
country_mapping = {
    'Canada': 'CA',
    'Finland': 'FI',
    'Italy': 'IT',
    'Kenya': 'KE',
    'Norway': 'NO',
    'Singapore': 'SG'
}
def is_holiday(date, country_name):
    country_code = country_mapping.get(country_name)
    if country_code:
        country_holidays = holidays.CountryHoliday(country_code)
        return date in country_holidays
    return False
def get_season(date):
    month = date.month
    day = date.day
    if (month == 12 and day >= 21) or (month in [1, 2]) or (month == 3 and day < 20):
        return 'Winter'
    elif (month == 3 and day >= 20) or (month in [4, 5]) or (month == 6 and day < 21):
        return 'Spring'
    elif (month == 6 and day >= 21) or (month in [7, 8]) or (month == 9 and day < 22):
        return 'Summer'
    else:
        return 'Fall'
train['is_holiday'] = train.apply(lambda row: is_holiday(row['date'], row['country']), axis=1)
train['season'] = train['date'].apply(lambda x: get_season(pd.to_datetime(x)))
train['saturday'] = train['date'].apply(lambda x: x.weekday() == 5)
train['sunday'] = train['date'].apply(lambda x: x.weekday() == 6)
test['is_holiday'] = test.apply(lambda row: is_holiday(row['date'], row['country']), axis=1)
test['season'] = test['date'].apply(lambda x: get_season(pd.to_datetime(x)))
test['saturday'] = test['date'].apply(lambda x: x.weekday() == 5)
test['sunday'] = test['date'].apply(lambda x: x.weekday() == 6)
train.head(), test.head()

(   id       date country              store             product  num_sold  \
 0   0 2010-01-01  Canada  Discount Stickers   Holographic Goose       NaN   
 1   1 2010-01-01  Canada  Discount Stickers              Kaggle     973.0   
 2   2 2010-01-01  Canada  Discount Stickers        Kaggle Tiers     906.0   
 3   3 2010-01-01  Canada  Discount Stickers            Kerneler     423.0   
 4   4 2010-01-01  Canada  Discount Stickers  Kerneler Dark Mode     491.0   
 
    is_holiday  season  saturday  sunday  
 0        True  Winter     False   False  
 1        True  Winter     False   False  
 2        True  Winter     False   False  
 3        True  Winter     False   False  
 4        True  Winter     False   False  ,
        id       date country              store             product  \
 0  230130 2017-01-01  Canada  Discount Stickers   Holographic Goose   
 1  230131 2017-01-01  Canada  Discount Stickers              Kaggle   
 2  230132 2017-01-01  Canada  Discount Stickers       

In [23]:
import pandas as pd
categorical_columns = ['country', 'store', 'product', 'is_holiday', 'season', 'saturday', 'sunday']
numerical_columns = ['num_sold']
num_buckets = 2000
def combined_hash(row):
    combined_string = ''.join([str(row[col]) for col in categorical_columns])
    x=  hash(combined_string)
    if(x<0):
        x%=(-num_buckets)
    else:
        x%=num_buckets
    return x 
print(train.shape)
train['hash_col'] = train.apply(combined_hash, axis=1)
print(train.shape)
print("unique values in hash_col: ", train['hash_col'].nunique())
print("range of values in hash_col: ", train['hash_col'].min(), train['hash_col'].max())
train.head()
print(test.shape)
test['hash_col'] = test.apply(combined_hash, axis=1)
print(test.shape)
print("unique values in hash_col: ", test['hash_col'].nunique())
print("range of values in hash_col: ", test['hash_col'].min(), test['hash_col'].max())
test.head()

#apply one hot encoding to the categorical columns
train = pd.get_dummies(train, columns=categorical_columns)
test = pd.get_dummies(test, columns=categorical_columns)
print(train.shape)
print(test.shape)

(230130, 10)
(230130, 11)
unique values in hash_col:  1531
range of values in hash_col:  -1997 1992
(98550, 9)
(98550, 10)
unique values in hash_col:  1430
range of values in hash_col:  -1997 1995
(230130, 28)
(98550, 27)


In [24]:
train['date'] = train['date'].apply(lambda x: x.toordinal())
train['sin_date'] = np.sin(2 * np.pi * train['date'] / 365)
train['cos_date'] = np.cos(2 * np.pi * train['date'] / 365)

test['date'] = test['date'].apply(lambda x: x.toordinal())
test['sin_date'] = np.sin(2 * np.pi * test['date'] / 365)
test['cos_date'] = np.cos(2 * np.pi * test['date'] / 365)

<h1> 0.6 mape w/ avg no find w/o avg </h1>

In [25]:
nan_df = None
if DATAIMPUTEAVG==False:
    nan_df = train[train['num_sold'].isnull()]
    print("shape of nan_df: ", nan_df.shape)
    #remove nan_df from train
    print("shape of train before removing nan_df", train.shape)
    train = train.drop(nan_df.index)
    print("shape of train after removing nan_df", train.shape)
else:
    train['num_sold'].fillna(train['num_sold'].mean(), inplace=True)
# print range of values in num_sold
print("range of values in num_sold: ", train['num_sold'].min(), train['num_sold'].max())

shape of nan_df:  (8871, 30)
shape of train before removing nan_df (230130, 30)
shape of train after removing nan_df (221259, 30)
range of values in num_sold:  5.0 5939.0


In [26]:
try:
    print(nan_df.shape)
    nan_df.head()
except:
    print("maybe dataimputer was on")

(8871, 30)


In [27]:
train.head()
traincopy = train.copy()
testcopy = test.copy()

class GlobalStats:
    def __init__(self, max_leafes=1500, max_depth=10, min_leaves=100, min_depth=3, min_lr=0.01, max_lr=0.3, trials=1, min_child_weight=1, max_child_weight=10, min_gamma=0, max_gamma=5, min_subsample=0.5, max_subsample=1.0, min_colsample_bytree=0.5, max_colsample_bytree=1.0, min_reg_alpha=1e-8, max_reg_alpha=1.0, min_reg_lambda=1e-8, max_reg_lambda=1.0, min_scale_pos_weight=0.1, max_scale_pos_weight=10.0,train_test_split=0.3):
        self.max_leafes = max_leafes
        self.max_depth = max_depth
        self.min_leaves = min_leaves
        self.min_depth = min_depth
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.trials = trials
        self.min_child_weight = min_child_weight
        self.max_child_weight = max_child_weight
        self.min_gamma = min_gamma
        self.max_gamma = max_gamma
        self.min_subsample = min_subsample
        self.max_subsample = max_subsample
        self.min_colsample_bytree = min_colsample_bytree
        self.max_colsample_bytree = max_colsample_bytree
        self.min_reg_alpha = min_reg_alpha
        self.max_reg_alpha = max_reg_alpha
        self.min_reg_lambda = min_reg_lambda
        self.max_reg_lambda = max_reg_lambda
        self.min_scale_pos_weight = min_scale_pos_weight
        self.max_scale_pos_weight = max_scale_pos_weight
        self.train_test_split = train_test_split

In [28]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

fig_hist = px.histogram(train, x='num_sold', nbins=10, title='Frequency of num_sold')
fig_hist.update_layout(xaxis_title='Number Sold', yaxis_title='Frequency')
fig_hist.show()
# Calculate value counts
value_counts = train['num_sold'].value_counts()
# Identify low-frequency values (less than the mean frequency)
mean_frequency = value_counts.mean()
low_freq_values = value_counts[value_counts < mean_frequency].index


In [29]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE

def binning_based_oversampling(data, target_col, bins=5, oversample_strategy='auto'):
    """
    Apply binning-based oversampling for regression tasks.

    Parameters:
        data (pd.DataFrame): Input DataFrame.
        target_col (str): Name of the target column.
        bins (int): Number of bins for target variable.
        oversample_strategy (str): Oversampling strategy for SMOTE.

    Returns:
        pd.DataFrame: Oversampled DataFrame.
    """
    # Separate features and target
    X = data.drop(columns=[target_col])
    y = data[target_col]

    # Bin the target variable into discrete categories
    bin_labels = range(bins)
    data['target_bin'], bin_edges = pd.qcut(y, q=bins, labels=bin_labels, retbins=True)

    oversampled_data = []
    for bin_label in bin_labels:
        bin_mask = data['target_bin'] == bin_label
        X_bin = X[bin_mask]
        y_bin = y[bin_mask]

        if len(y_bin) < 2:  # Skip bins with too few samples
            oversampled_data.append(pd.concat([X_bin, y_bin], axis=1))
            continue

        smote = SMOTE(sampling_strategy=oversample_strategy, k_neighbors=min(5, len(y_bin) - 1))
        X_resampled, y_bin_resampled = smote.fit_resample(X_bin, y_bin.index)

        # Map the resampled target bin back to the original range
        y_resampled = np.random.uniform(
            low=bin_edges[bin_label],
            high=bin_edges[bin_label + 1],
            size=len(y_bin_resampled)
        )

        oversampled_bin = pd.concat(
            [pd.DataFrame(X_resampled, columns=X.columns), pd.Series(y_resampled, name=target_col)], axis=1
        )
        oversampled_data.append(oversampled_bin)

    # Combine oversampled data
    oversampled_data = pd.concat(oversampled_data, axis=0).reset_index(drop=True)
    return oversampled_data

# Example usage
# Assuming `train` is your DataFrame and 'num_sold' is the target column
if DATABALANCE:
    for i in range(1):
        train = binning_based_oversampling(train, target_col='num_sold', bins=7)

In [31]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

fig_hist = px.histogram(train, x='num_sold', nbins=10, title='Frequency of num_sold')
fig_hist.update_layout(xaxis_title='Number Sold', yaxis_title='Frequency')
fig_hist.show()
# Calculate value counts
value_counts = train['num_sold'].value_counts()
# Identify low-frequency values (less than the mean frequency)
mean_frequency = value_counts.mean()
low_freq_values = value_counts[value_counts < mean_frequency].index


In [30]:
traincopy2 = train.copy()
testcopy2 = test.copy()

In [ ]:
train = traincopy2.copy()
test = testcopy2.copy()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Calculate the bin edges
num_bins = 10
counts, bin_edges = np.histogram(train['num_sold'], bins=num_bins)
# Separate the data into bins
bins = []
for i in range(len(bin_edges) - 1):
    bin_data = train[(train['num_sold'] >= bin_edges[i]) & (train['num_sold'] < bin_edges[i + 1])]
    bins.append(bin_data)
# Print the bins
for i, bin_data in enumerate(bins):
    print(f"Bin {i + 1}:")
    print(bin_data.shape)
    print()
# Double the frequency of bins 5 through 10
for i in range(3, len(bins)-1):  # Bins are 0-indexed, so bins 5 through 10 are indices 4 through 9
    bins[i] = pd.concat([bins[i], bins[i]])
bins[len(bins)-1] = pd.concat([bins[len(bins)-1], bins[len(bins)-1], bins[len(bins)-1], bins[len(bins)-1]])

# Combine all bins back into a single DataFrame
print(f' shape of train before augmentation: {train.shape}')
train_augmented = pd.concat(bins, axis=0).reset_index(drop=True)
print(f' shape of train after augmentation: {train_augmented.shape}')
# Plot histogram after augmentation
fig_hist = px.histogram(train_augmented, x='num_sold', nbins=num_bins, title='Frequency of num_sold (After Augmentation)')
fig_hist.update_layout(xaxis_title='Number Sold', yaxis_title='Frequency')
fig_hist.show()

value_counts = train_augmented['num_sold'].value_counts()
# Identify low-frequency values (less than the mean frequency)
mean_frequency = value_counts.mean()
low_freq_values = value_counts[value_counts < mean_frequency].index


In [ ]:


def objective(trial, train, model_type, globalobj):
    # Make a copy of train
    train.drop(columns=['date', 'id'], inplace=True)
    starttime = time.time()
    if model_type == 'cat':
        globalobj.max_depth = 16

    # Define hyperparameters to tune
    if model_type == 'xgb':
        param = {
            'n_estimators': trial.suggest_int('n_estimators', globalobj.min_leaves, globalobj.max_leafes),
            'max_depth': trial.suggest_int('max_depth', globalobj.min_depth, globalobj.max_depth),
            'learning_rate': trial.suggest_float('learning_rate', globalobj.min_lr, globalobj.max_lr, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', globalobj.min_child_weight, globalobj.max_child_weight),
            'gamma': trial.suggest_float('gamma', globalobj.min_gamma, globalobj.max_gamma),
            'subsample': trial.suggest_float('subsample', globalobj.min_subsample, globalobj.max_subsample),
            'colsample_bytree': trial.suggest_float('colsample_bytree', globalobj.min_colsample_bytree, globalobj.max_colsample_bytree),
            'reg_alpha': trial.suggest_float('reg_alpha', globalobj.min_reg_alpha, globalobj.max_reg_alpha, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', globalobj.min_reg_lambda, globalobj.max_reg_lambda, log=True),
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', globalobj.min_scale_pos_weight, globalobj.max_scale_pos_weight),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
    elif model_type == 'cat':
        param = {
            'iterations': trial.suggest_int('iterations', globalobj.min_leaves, globalobj.max_leafes),
            'depth': trial.suggest_int('depth', globalobj.min_depth, globalobj.max_depth),
            'learning_rate': trial.suggest_float('learning_rate', globalobj.min_lr, globalobj.max_lr, log=True),
            'random_seed': 42,
            'silent': True
        }
        model = cat.CatBoostRegressor(**param)
    elif model_type == 'lgbm':
        param = {
            'n_estimators': trial.suggest_int('n_estimators', globalobj.min_leaves, globalobj.max_leafes),
            'max_depth': trial.suggest_int('max_depth', globalobj.min_depth, globalobj.max_depth),
            'learning_rate': trial.suggest_float('learning_rate', globalobj.min_lr, globalobj.max_lr, log=True),
            'random_state': 42,
            'verbose': -1  # Suppress warnings
        }
        model = lgb.LGBMRegressor(**param)
    else:
        raise ValueError("Invalid model type. Choose from 'xgb', 'cat', or 'lgbm'.")

    # Split the data
    train = train.astype('int64')
    X_train, X_valid, y_train, y_valid = train_test_split(train.drop(['num_sold'], axis=1), train['num_sold'], test_size=globalobj.train_test_split, random_state=42)
    
    # Combine X_train and y_train into a single DataFrame and reset indices
    train_combined = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)
    
    # Debugging information
    print(f"train_combined shape: {train_combined.shape}")
    print(f"train_combined columns: {train_combined.columns}")
    
    # Apply SMOGN
    try:
        train_resampled = smogn.smoter(
            data=train_combined,
            y='num_sold'
        )
    except Exception as e:
        print(f"Error during SMOGN: {e}")
        return float('inf')
    
    # Separate the resampled data back into X and y
    X_train_resampled = train_resampled.drop(columns=['num_sold'])
    y_train_resampled = train_resampled['num_sold']
    
    # Train the model
    model.fit(X_train_resampled, y_train_resampled)
    predictions = model.predict(X_valid)
    mape = mean_absolute_percentage_error(y_valid, predictions)
    print(f'time for a trial with {model_type} is {time.time()-starttime}')
    return mape

def train_and_predict(train, model_type, test, globalobj):
    # Optimize hyperparameters with Optuna
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, train.copy(), model_type, globalobj), n_trials=globalobj.trials)
    print(f'Best hyperparameters: {study.best_params}')
    # Train the model with the best hyperparameters
    best_params = study.best_params
    print(best_params)
    if model_type == 'xgb':
        model = xgb.XGBRegressor(**best_params)
    elif model_type == 'cat':
        model = cat.CatBoostRegressor(**best_params)
    elif model_type == 'lgbm':
        model = lgb.LGBMRegressor(**best_params)

    date_column = train['date']
    id_column = train['id']
    train.drop(columns=['date', 'id'], inplace=True)

    if model_type == 'cat' or model_type == 'xgb':
        model.fit(train.drop(['num_sold'], axis=1), train['num_sold'], verbose=False)
    else:
        model.fit(train.drop(['num_sold'], axis=1), train['num_sold'])

    train['date'] = date_column
    train['id'] = id_column
    date_column = train['date']
    id_column = train['id']
    num_sold_actual = train['num_sold']
    train.drop(columns=['date', 'id', 'num_sold'], inplace=True)

    predictions = model.predict(train)

    decimal_found = False
    fractional_sum = 0
    for pred in predictions:
        fractional_part = pred - int(pred)
        if fractional_part != 0:
            decimal_found = True
            fractional_sum += fractional_part

    if decimal_found:
        print("There was a decimal number in the predictions.")
    print(f'Sum of all fractional parts: {fractional_sum}')
    predictions = [round(pred) for pred in predictions]
    train['date'] = date_column
    train['id'] = id_column
    train['num_sold'] = num_sold_actual
    mape = mean_absolute_percentage_error(num_sold_actual, predictions)
    print(f'Mean Absolute Percentage Error: {mape}')

    # Plotting
    plt.figure(figsize=(15, 15))
    plt.subplot(3, 1, 1)
    red = (1, 0, 0, 1)  # Red with full opacity
    green = (0, 1, 0, 0.5)  
    import plotly.graph_objs as go
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=1, cols=1, subplot_titles=("Actual vs Predicted Number Sold"))
    red = 'rgba(255, 0, 0, 1)'  # Red with full opacity
    green = 'rgba(0, 255, 0, 0.5)'  # Green with 50% opacity
    fig.add_trace(go.Scatter(x=train['id'], y=num_sold_actual, mode='lines', name='Actual num_sold', line=dict(color=red)), row=1, col=1)
    fig.add_trace(go.Scatter(x=train['id'], y=predictions, mode='lines', name='Predicted num_sold', line=dict(color=green, dash='dash')), row=1, col=1)
    fig.update_layout(height=500, width=1200, title_text="Number Sold Analysis", showlegend=True)
    fig.show()

    test_id = test['id']
    test_date = test['date']
    test_pred = model.predict(test.drop(['date', 'id'], axis=1))
    test_pred = [round(pred) for pred in test_pred]

    # Add the test_id and test_pred to a new dataframe
    test_df = pd.DataFrame({'id': test_id, 'num_sold': test_pred})

    # Save the test_df to a CSV file
    if LOCAL:
        test_df.to_csv(f'./kaggle/working/submission{model_type}.csv', index=False)
    else:
        test_df.to_csv(f'/kaggle/working/submission{model_type}.csv', index=False)

    train = train.copy()
    test = test.copy()


# train_and_predict(train_data, 'xgb', test_data, globalobj)

In [ ]:
# Define the GlobalStats object with the specified parameters
# Example usage
globalobj = GlobalStats(
    max_leafes=1500,
    max_depth=10,
    min_leaves=100,
    min_depth=3,
    min_lr=0.01,
    max_lr=0.3,
    trials=100,
    min_child_weight=1,
    max_child_weight=10,
    min_gamma=0,
    max_gamma=20,
    min_subsample=0.3,
    max_subsample=1.0,
    min_colsample_bytree=0.5,
    max_colsample_bytree=1.0,
    min_reg_alpha=1e-8,
    max_reg_alpha=1.0,
    min_reg_lambda=1e-8,
    max_reg_lambda=1.0,
    min_scale_pos_weight=0.1,
    max_scale_pos_weight=10.0,
    train_test_split=0.2
)

#check if train and test have date and id columns
if 'date' in train.columns and 'id' in train.columns and 'date' in test.columns and 'id' in test.columns:
    print("train and test have date and id columns")
else:
    print("train and test do not have date and id columns")
train_and_predict(train, 'xgb',test,globalobj)


In [ ]:
globalobjcat = GlobalStats(
    max_leafes=1500,
    min_leaves=100,
    max_depth=25,
    min_depth=3,
    min_lr=1e-5,
    max_lr=0.9,
    trials=50,
    min_child_weight=1,
    max_child_weight=10,
    min_gamma=0,
    max_gamma=5,
    min_subsample=0.5,
    max_subsample=1.0,
    min_colsample_bytree=0.5,
    max_colsample_bytree=1.0,
    min_reg_alpha=1e-8,
    max_reg_alpha=1.0,
    min_reg_lambda=1e-8,
    max_reg_lambda=1.0
)

train_and_predict(train, 'cat',test,globalobjcat)

In [ ]:
globalobjlgb = GlobalStats(
    max_leafes=1500,
    min_leaves=100,
    max_depth=25,
    min_depth=3,
    min_lr=1e-5,
    max_lr=0.9,
    trials=50,
    min_child_weight=1,
    max_child_weight=10,
    min_gamma=0,
    max_gamma=5,
    min_subsample=0.5,
    max_subsample=1.0,
    min_colsample_bytree=0.5,
    max_colsample_bytree=1.0,
    min_reg_alpha=1e-8,
    max_reg_alpha=1.0,
    min_reg_lambda=1e-8,
    max_reg_lambda=1.0
)

train_and_predict(train, 'lgbm',test,globalobjlgb)
